# Integración de los datos

In [50]:
import sys, os, duckdb
import pandas as pd
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

features_dir = "../../data/features/"
db_path = "../../data/processed/olist_analytics.duckdb"
os.makedirs("../../data/processed/", exist_ok=True)

con = duckdb.connect(db_path, read_only=False)

orders  = pd.read_csv(features_dir + "features_orders.csv", parse_dates=[
    "order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
    "order_delivered_customer_date","order_estimated_delivery_date"
])
items   = pd.read_csv(features_dir + "features_items_agg.csv")
reviews = pd.read_csv(features_dir + "features_reviews.csv", parse_dates=[
    "review_creation_date","review_answer_timestamp"
])
sellers = pd.read_csv("../../data/processed/olist_sellers_clean.csv")
order_items = pd.read_csv("../../data/processed/olist_order_items_clean.csv")

con.register("orders_df", orders)
con.register("items_df", items)
con.register("reviews_df", reviews)
con.register("sellers_df", sellers)
con.register("order_items_df", order_items)

In [47]:
# VISTA VENTAS
con.execute("""
CREATE OR REPLACE VIEW vw_sales AS
SELECT
    i.order_id,
    i.item_count AS items_per_order,
    i.order_price_total AS price,
    i.order_freight_total AS freight_value,
    (i.order_price_total + i.order_freight_total) AS order_total_value,
    i.seller_count,
    o.order_year,
    o.order_month,
    o.order_dow
FROM items_df i
LEFT JOIN orders_df o USING(order_id);
""")

In [46]:
# VISTA LOGÍSTICA
con.execute("""
CREATE OR REPLACE VIEW vw_logistics AS
SELECT
    order_id,
    delivery_days,
    delay_vs_estimated,
    on_time
FROM orders_df;
""")

In [48]:
# VISTA SATISFACCION
con.execute("""
CREATE OR REPLACE VIEW vw_customer_satisfaction AS
SELECT
    order_id,
    review_score
FROM reviews_df;
""")

In [51]:
# VISTA VENDEDORES
con.execute("""
CREATE OR REPLACE VIEW vw_sellers AS
WITH base AS (
    SELECT
        s.seller_id,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        COUNT(oi.order_item_id) AS total_items,
        SUM(oi.price + oi.freight_value) AS total_gmv,
        AVG(o.delivery_days) AS avg_delivery_days,
        AVG(o.delay_vs_estimated) AS avg_delay,
        AVG(r.review_score) AS avg_review_score
    FROM sellers_df s
    LEFT JOIN order_items_df oi ON s.seller_id = oi.seller_id
    LEFT JOIN orders_df o ON oi.order_id = o.order_id
    LEFT JOIN reviews_df r ON oi.order_id = r.order_id
    GROUP BY s.seller_id
)
SELECT * FROM base;
""")

# Resultados

In [58]:
for v in ["vw_logistics", "vw_sales", "vw_customer_satisfaction", "vw_sellers"]:
    print(f"\n--- {v} ---")
    display(con.execute(f"SELECT * FROM {v} LIMIT 10").fetchdf())



--- vw_logistics ---


,order_id,delivery_days,delay_vs_estimated,on_time
0,10a045cdf6a5650c21e9cfeb60384c16,NaN,NaN,False
1,b059ee4de278302d550a3035c4cdb740,NaN,NaN,False
2,a2ac6dad85cf8af5b0afb510a240fe8c,NaN,NaN,False
3,616fa7d4871b87832197b2a137a115d2,NaN,NaN,False
4,392ed9afd714e3c74767d0c4d3e3f477,NaN,NaN,False
5,869997fbe01f39d184956b5c6bccfdbe,NaN,NaN,False
6,5aac76cf7b07dd06fa4d50bf461d2f40,NaN,NaN,False
7,ed3efbd3a87bea76c2812c66a0b32219,NaN,NaN,False
8,bd35b677fd239386e9861d11ae98ab56,NaN,NaN,False
9,ea844c92cf978ea23321fa7fe5871761,NaN,NaN,False



--- vw_sales ---


,order_id,items_per_order,price,freight_value,order_total_value,seller_count,order_year,order_month,order_dow
0,54282e97f61c23b78330c15b154c867d,1,145.00,21.46,166.46,1,2018,2018-09,0
1,35a972d7f8436f405b56e36add1a7140,1,84.99,8.76,93.75,1,2018,2018-08,2
2,03ef5dedbe7492bdae72eec50764c43f,1,24.90,8.33,33.23,1,2018,2018-08,2
3,168626408cb32af0ffaf76711caae1dc,1,45.90,15.39,61.29,1,2018,2018-08,2
4,0b223d92c27432930dfe407c6aea3041,2,418.00,92.96,510.96,1,2018,2018-08,2
5,52018484704db3661b98ce838612b507,1,63.90,9.20,73.10,1,2018,2018-08,2
6,d03ca98f59480e7e76c71fa83ecd8fb6,1,109.90,9.52,119.42,1,2018,2018-08,2
7,d70442bc5e3cb7438da497cc6a210f80,1,6.90,7.39,14.29,1,2018,2018-08,2
8,912859fef5a0bd5059b6d48fa79d121a,1,169.80,8.45,178.25,1,2018,2018-08,2
9,fb393211459aac00af932cd7ab4fa2cc,1,99.00,7.95,106.95,1,2018,2018-08,2



--- vw_customer_satisfaction ---


,order_id,review_score
0,bd2c2c3a4d59e68fb14a526745572883,4
1,529a65336debb1c3e3327d7d67dc733d,5
2,b0e9288a209f5ec50391c140dba4c91f,5
3,d02b32c3bcfb76481817b2222b990e84,5
4,8506571faf231af2bd4e43d1ba47dce8,5
5,0a06917fb1c4d93dddc71d9ca7dd8790,5
6,d2c57d721916c36343711be4e205ad5e,4
7,124223dc899eca6ac54187a54f1e3632,5
8,6ea4cd6f1ec3ef5b3dfeb0c90093a5d0,5
9,c3a4dcc5c4ae575cb04444cf1b3f0a09,5



--- vw_sellers ---


,seller_id,total_orders,total_items,total_gmv,avg_delivery_days,avg_delay,avg_review_score
0,6560211a19b47992c3666cc44a7e94c0,1853,2030,150896.75,9.539235,-11.008311,3.910990
1,ccf8813e5a7d6c84d865cd38bfc2b130,26,26,3051.29,11.430515,-12.802580,4.384615
2,d9d43faf741ceaa8da52fdbb81d9e0e3,6,8,421.25,9.545696,-15.188873,3.500000
3,8ea394aed8138685abe1eb9f25e1021d,2,2,1417.60,11.290255,-8.751157,4.000000
4,8b321bb669392f5163d04c59e235e066,943,1018,31692.05,12.616899,-9.298743,3.995069
5,0bae85eb84b9fb3bd773911e89288d54,136,145,10279.84,10.260313,-14.584851,4.206897
6,7d13fca15225358621be4086e1eb0964,556,569,116701.56,15.645123,-10.419892,4.014159
7,f593898ec748b7a8cb81fc04edafd98a,19,22,3314.15,17.617965,-13.227299,3.500000
8,5a8e7d5003a1f221f9e1d6e411de7c23,139,181,13866.19,10.467906,-11.780227,4.172222
9,dbc22125167c298ef99da25668e1011f,406,429,41789.16,11.405500,-13.172511,4.226636


# Guardado

In [59]:
output_dir = "../../data/views/"
os.makedirs(output_dir, exist_ok=True)

for v in ["vw_logistics", "vw_sales", "vw_customer_satisfaction", "vw_sellers"]:
    df = con.execute(f"SELECT * FROM {v}").fetchdf()
    df.to_csv(output_dir + f"{v}.csv", index=False)

print("Views exported to:", output_dir)


Views exported to: ../../data/views/
